In poctest.ipynb werden de modellen getest. Nu word hierop verder gebouwd om tot een resultaat te bekomen waardat bij een foto van een winkelschap alle producten herkent worden en correct beschreven worden.

De structuur die we volgen voor de pipeline op te stellen is als volgt:
- Image
-  ↓
- YOLO herkent waar producten staan
-  ↓
- Crop de producten die gevonden worden
-  ↓
- CLIP + Florence-2 maken weight voor elke crop
-  ↓
- FAISS zoekt naar afbeelding met similar weight in de openfoodfacts database zodat het zelfde product gevonden kan worden

Imports

In [42]:
from ultralytics import YOLO, RTDETR
from transformers import Owlv2Processor, Owlv2ForObjectDetection, MobileViTImageProcessor, MobileViTModel, AutoProcessor, AutoModelForCausalLM, CLIPProcessor, CLIPModel
import torch
from nanoowl.owl_predictor import OwlPredictor
from PIL import Image
import cv2
import sys
import types
import importlib.machinery
import os
import clip
import pandas as pd
import duckdb
import numpy as np
import faiss
import uuid
from datetime import datetime
import requests

Foodfacts database

In [43]:
df_foodfacts = duckdb.query("""
    SELECT code, product_name, brands, images, countries_tags, ingredients_analysis_tags, nutriments
    FROM 'C:\\Users\\krist\\Documents\\Hogent_IT\\3de_jaar\\Bachelorproef\\bachproef26\\latex-hogent-bachproef\\bachproef\\datasets\\food.parquet'
    WHERE list_contains(countries_tags, 'en:belgium')
    AND brands IS NOT NULL
    AND images IS NOT NULL
""").to_df()

print(df_foodfacts.columns)
print(len(df_foodfacts))

Index(['code', 'product_name', 'brands', 'images', 'countries_tags',
       'ingredients_analysis_tags', 'nutriments'],
      dtype='str')
65999


In [46]:
def build_image_url(code, key, rev, size="400"):
    code = str(code).zfill(13)

    prefix = code[:-4]
    suffix = code[-4:]

    parts = [prefix[i:i+3] for i in range(0, len(prefix), 3) if prefix[i:i+3]]
    parts.append(suffix)

    path = "/".join(parts)
    url = f"https://images.openfoodfacts.org/images/products/{path}/{key}.{rev}.{size}.jpg"
    print(url)
    return url

def extract_front_image(row):
    images = row["images"]

    if images is None:
        return None

    priority = ["front_nl", "front_fr", "front_en", "front"]

    for pref in priority:
        for img in images:
            key = img.get("key")
            rev = img.get("rev")

            if key and rev and key.startswith(pref):
                return build_image_url(row["code"], key, rev)

    return None